In [1]:
pip install pandas mysql-connector-python openpyxl scikit-learn


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import os
import mysql.connector

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

In [3]:
folder_path = "sales"

files = [f for f in os.listdir(folder_path) if f.endswith(".xlsx")]

dataframes = {}

for file in files:
    path = os.path.join(folder_path, file)
    df = pd.read_excel(path)
    
    table_name = file.replace(".xlsx", "")
    dataframes[table_name] = df

In [4]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="1234",
    database="sales_db"   # تأكد أنها موجودة
)

cursor = conn.cursor()

In [5]:
def build_pipeline(df):
    
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns
    cat_cols = df.select_dtypes(include=['object']).columns

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols)
        ]
    )

    return preprocessor

In [8]:
for table_name, df in dataframes.items():

    print(f"Processing {table_name} ...")

    pipeline = build_pipeline(df)

    transformed_array = pipeline.fit_transform(df)

    # get feature names (important)
    feature_names = pipeline.get_feature_names_out()

    transformed_df = pd.DataFrame(transformed_array, columns=feature_names)

    # create table in MySQL
    cursor.execute(f"DROP TABLE IF EXISTS {table_name}_processed")

    columns_sql = ", ".join([f"`{col}` FLOAT" for col in transformed_df.columns])

    create_table_query = f"""
    CREATE TABLE {table_name}_processed (
        id INT AUTO_INCREMENT PRIMARY KEY,
        {columns_sql}
    )
    """

    cursor.execute(create_table_query)

    # insert data
    for _, row in transformed_df.iterrows():
        values = tuple(row)
        placeholders = ", ".join(["%s"] * len(values))

        insert_query = f"""
        INSERT INTO {table_name}_processed 
        ({", ".join(transformed_df.columns)})
        VALUES ({placeholders})
        """

        cursor.execute(insert_query, values)

    conn.commit()

    print(f"{table_name} saved successfully ✔")

In [9]:
cursor.close()
conn.close()

In [10]:
cursor.execute("SHOW TABLES")
tables = cursor.fetchall()

print(tables)

ProgrammingError: 2055: Cursor is not connected

In [11]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="1234",
    database="sales_db"
)

cursor = conn.cursor()

In [12]:
print(conn.is_connected())

True


In [13]:
cursor.execute("SHOW TABLES")
tables = cursor.fetchall()

print(tables)

[]


In [14]:
cursor.execute("SELECT DATABASE();")
print(cursor.fetchone())

('sales_db',)


In [15]:
import os

folder_path = "sales"

files = [f for f in os.listdir(folder_path) if f.endswith(".xlsx")]

print(files)

[]


In [16]:
import os

print(os.getcwd())

C:\Users\DELL\wajd-mq


In [17]:
folder_path = r"C:\Users\DELL\wajd-mq\sales"

In [18]:
import os

files = [f for f in os.listdir(folder_path) if f.endswith(".xlsx")]

print(files)

[]


In [4]:
print(os.listdir(r"C:\Users\DELL\wajd-mq\sales"))

['store_sales_1.csv', 'store_sales_2.csv', 'store_sales_3.csv']


In [2]:
import os

In [5]:
import pandas as pd
import os

folder_path = r"C:\Users\DELL\wajd-mq\sales"

files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

print(files)

['store_sales_1.csv', 'store_sales_2.csv', 'store_sales_3.csv']


In [6]:
dataframes = {}

for file in files:
    
    path = os.path.join(folder_path, file)

    df = pd.read_csv(path)

    table_name = file.replace(".csv", "")

    dataframes[table_name] = df

    print(f"{table_name} loaded ✔")

store_sales_1 loaded ✔
store_sales_2 loaded ✔
store_sales_3 loaded ✔


In [8]:
import mysql.connector

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# =========================
# MYSQL CONNECTION
# =========================

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="1234"
)

cursor = conn.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS sales_db")
cursor.execute("USE sales_db")

# =========================
# PROCESS EACH FILE
# =========================

for table_name, df in dataframes.items():

    print(f"Processing {table_name}")

    # detect columns
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns
    cat_cols = df.select_dtypes(include=['object']).columns

    # pipelines
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols)
        ]
    )

    # transform
    transformed_array = preprocessor.fit_transform(df)

    # feature names
    feature_names = preprocessor.get_feature_names_out()

    transformed_df = pd.DataFrame(
        transformed_array,
        columns=feature_names
    )

    # clean names
    transformed_df.columns = (
        transformed_df.columns
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )

    # =========================
    # CREATE TABLE
    # =========================

    cursor.execute(f"DROP TABLE IF EXISTS {table_name}_processed")

    columns_sql = ", ".join(
        [f"`{col}` FLOAT" for col in transformed_df.columns]
    )

    create_query = f"""
    CREATE TABLE {table_name}_processed (
        id INT AUTO_INCREMENT PRIMARY KEY,
        {columns_sql}
    )
    """

    cursor.execute(create_query)

    conn.commit()

    print(f"Table {table_name}_processed created ✔")

Processing store_sales_1


C:\Users\DELL\AppData\Local\Temp\ipykernel_11408\3239711707.py:33: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns


ValueError: Shape of passed values is (100, 1), indices imply (100, 289)

In [11]:
# =========================================================
# SALES CSV FILES -> PREPROCESSING -> MYSQL DATABASE
# =========================================================

import pandas as pd
import os
import mysql.connector

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# =========================================================
# 1) READ CSV FILES
# =========================================================

folder_path = r"C:\Users\DELL\wajd-mq\sales"

files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

print("Files Found:")
print(files)

dataframes = {}

for file in files:

    path = os.path.join(folder_path, file)

    df = pd.read_csv(path)

    table_name = file.replace(".csv", "")

    dataframes[table_name] = df

    print(f"{table_name} loaded successfully ✔")


# =========================================================
# 2) MYSQL CONNECTION
# =========================================================

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="1234"   # <-- ضع الباسورد هنا
)

cursor = conn.cursor()

# create database if not exists
cursor.execute("CREATE DATABASE IF NOT EXISTS sales_db")

# use database
cursor.execute("USE sales_db")

print("Connected to MySQL ✔")


# =========================================================
# 3) PROCESS EACH DATAFRAME
# =========================================================

for table_name, df in dataframes.items():

    print(f"\nProcessing {table_name} ...")

    # =====================================================
    # Detect numeric and categorical columns
    # =====================================================

    num_cols = df.select_dtypes(
        include=['int64', 'float64']
    ).columns

    cat_cols = df.select_dtypes(
        include=['object', 'string']
    ).columns

    # =====================================================
    # Numeric Pipeline
    # =====================================================

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    # =====================================================
    # Categorical Pipeline
    # =====================================================

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ])

    # =====================================================
    # Column Transformer
    # =====================================================

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols)
        ]
    )

    # =====================================================
    # Transform Data
    # =====================================================

    transformed_array = preprocessor.fit_transform(df)

    # get feature names
    feature_names = preprocessor.get_feature_names_out()

    # create transformed dataframe
    transformed_df = pd.DataFrame(
        transformed_array,
        columns=feature_names
    )

 # =====================================================
# Clean Column Names
# =====================================================

    transformed_df.columns = (
    transformed_df.columns
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.replace("/", "_")
)

# remove duplicate columns
    transformed_df = transformed_df.loc[:, ~transformed_df.columns.duplicated()]
    # =====================================================
    # Create MySQL Table
    # =====================================================

    processed_table = f"{table_name}_processed"

    cursor.execute(f"DROP TABLE IF EXISTS {processed_table}")

    columns_sql = ", ".join([
        f"`{col}` FLOAT"
        for col in transformed_df.columns
    ])

    create_query = f"""
    CREATE TABLE {processed_table} (
        id INT AUTO_INCREMENT PRIMARY KEY,
        {columns_sql}
    )
    """

    cursor.execute(create_query)

    conn.commit()

    print(f"Table {processed_table} created ✔")

    # =====================================================
    # Insert Data into MySQL
    # =====================================================

    for _, row in transformed_df.iterrows():

        values = tuple(row)

        placeholders = ", ".join(["%s"] * len(values))

        insert_query = f"""
        INSERT INTO {processed_table}
        ({", ".join([f"`{col}`" for col in transformed_df.columns])})
        VALUES ({placeholders})
        """

        cursor.execute(insert_query, values)

    conn.commit()

    print(f"Data inserted into {processed_table} ✔")


# =========================================================
# 4) SHOW TABLES
# =========================================================

cursor.execute("SHOW TABLES")

tables = cursor.fetchall()

print("\nTables in sales_db:")
print(tables)

# =========================================================
# 5) CLOSE CONNECTION
# =========================================================

cursor.close()
conn.close()

print("\nDONE SUCCESSFULLY 🚀")

Files Found:
['store_sales_1.csv', 'store_sales_2.csv', 'store_sales_3.csv']
store_sales_1 loaded successfully ✔
store_sales_2 loaded successfully ✔
store_sales_3 loaded successfully ✔
Connected to MySQL ✔

Processing store_sales_1 ...


ProgrammingError: 1060 (42S21): Duplicate column name 'cat__CurrencyType_Usd'

In [12]:
import pandas as pd
import os
import mysql.connector

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# =========================================================
# 1) LOAD CSV FILES
# =========================================================

folder_path = r"C:\Users\DELL\wajd-mq\sales"

files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

print("Files Found:", files)

dataframes = {}

for file in files:
    path = os.path.join(folder_path, file)
    df = pd.read_csv(path)

    table_name = file.replace(".csv", "")
    dataframes[table_name] = df

    print(f"{table_name} loaded ✔")


# =========================================================
# 2) MYSQL CONNECTION
# =========================================================

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="1234"
)

cursor = conn.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS sales_db")
cursor.execute("USE sales_db")

print("Connected to MySQL ✔")


# =========================================================
# 3) COMBINE DATA (IMPORTANT FIX)
#    → to prevent duplicate encoding issues
# =========================================================

all_df = pd.concat(dataframes.values(), ignore_index=True)

# detect columns ONCE
num_cols = all_df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = all_df.select_dtypes(include=['object', 'string']).columns


# =========================================================
# 4) PIPELINE (FIT ONLY ONCE)
# =========================================================

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

# fit once
preprocessor.fit(all_df)


# =========================================================
# 5) PROCESS EACH FILE
# =========================================================

for table_name, df in dataframes.items():

    print(f"\nProcessing {table_name} ...")

    transformed_array = preprocessor.transform(df)

    feature_names = preprocessor.get_feature_names_out()

    transformed_df = pd.DataFrame(
        transformed_array,
        columns=feature_names
    )

    # CLEAN COLUMN NAMES
    transformed_df.columns = (
        transformed_df.columns
        .str.replace(" ", "_")
        .str.replace("-", "_")
        .str.replace("/", "_")
    )

    # REMOVE DUPLICATE COLUMNS (IMPORTANT FIX)
    transformed_df = transformed_df.loc[
        :, ~transformed_df.columns.duplicated()
    ]

    # =====================================================
    # CREATE MYSQL TABLE
    # =====================================================

    processed_table = f"{table_name}_processed"

    cursor.execute(f"DROP TABLE IF EXISTS {processed_table}")

    columns_sql = ", ".join(
        [f"`{col}` FLOAT" for col in transformed_df.columns]
    )

    create_query = f"""
    CREATE TABLE {processed_table} (
        id INT AUTO_INCREMENT PRIMARY KEY,
        {columns_sql}
    )
    """

    cursor.execute(create_query)
    conn.commit()

    print(f"Table {processed_table} created ✔")


    # =====================================================
    # INSERT DATA
    # =====================================================

    for _, row in transformed_df.iterrows():

        values = tuple(row)

        placeholders = ", ".join(["%s"] * len(values))

        insert_query = f"""
        INSERT INTO {processed_table}
        ({", ".join([f"`{col}`" for col in transformed_df.columns])})
        VALUES ({placeholders})
        """

        cursor.execute(insert_query, values)

    conn.commit()

    print(f"Data inserted into {processed_table} ✔")


# =========================================================
# 6) SHOW TABLES
# =========================================================

cursor.execute("SHOW TABLES")
print("\nTables in sales_db:")
print(cursor.fetchall())


# =========================================================
# 7) CLOSE CONNECTION
# =========================================================

cursor.close()
conn.close()

print("\nDONE SUCCESSFULLY 🚀")

Files Found: ['store_sales_1.csv', 'store_sales_2.csv', 'store_sales_3.csv']
store_sales_1 loaded ✔
store_sales_2 loaded ✔
store_sales_3 loaded ✔
Connected to MySQL ✔

Processing store_sales_1 ...


ProgrammingError: 1060 (42S21): Duplicate column name 'cat__CurrencyType_Usd'

In [13]:
import pandas as pd
import os
import mysql.connector

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# =========================================================
# 1) LOAD DATA
# =========================================================

folder_path = r"C:\Users\DELL\wajd-mq\sales"

files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

print("Files Found:", files)

dataframes = {}

for file in files:
    path = os.path.join(folder_path, file)
    df = pd.read_csv(path)

    table_name = file.replace(".csv", "")
    dataframes[table_name] = df

    print(f"{table_name} loaded ✔")


# =========================================================
# 2) MYSQL CONNECTION
# =========================================================

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="1234"
)

cursor = conn.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS sales_db")
cursor.execute("USE sales_db")

print("Connected ✔")


# =========================================================
# 3) PROCESS EACH FILE (NO ONE-HOT)
# =========================================================

for table_name, df in dataframes.items():

    print(f"\nProcessing {table_name} ...")

    # detect columns
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns
    cat_cols = df.select_dtypes(include=['object', 'string']).columns

    # fill missing values
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    df[cat_cols] = df[cat_cols].fillna("Unknown")

    # =====================================================
    # SIMPLE LABEL ENCODING (NO ONE-HOT)
    # =====================================================

    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

    # =====================================================
    # SCALING NUMERIC ONLY
    # =====================================================

    scaler = StandardScaler()
    df[num_cols] = scaler.fit_transform(df[num_cols])

    # =====================================================
    # CREATE MYSQL TABLE
    # =====================================================

    processed_table = f"{table_name}_processed"

    cursor.execute(f"DROP TABLE IF EXISTS {processed_table}")

    columns_sql = ", ".join([
        f"`{col}` FLOAT"
        for col in df.columns
    ])

    create_query = f"""
    CREATE TABLE {processed_table} (
        id INT AUTO_INCREMENT PRIMARY KEY,
        {columns_sql}
    )
    """

    cursor.execute(create_query)
    conn.commit()

    print(f"Table {processed_table} created ✔")

    # =====================================================
    # INSERT DATA
    # =====================================================

    for _, row in df.iterrows():

        values = tuple(row)
        placeholders = ", ".join(["%s"] * len(values))

        insert_query = f"""
        INSERT INTO {processed_table}
        ({", ".join([f"`{col}`" for col in df.columns])})
        VALUES ({placeholders})
        """

        cursor.execute(insert_query, values)

    conn.commit()

    print(f"Data inserted ✔")


# =========================================================
# 4) DONE
# =========================================================

cursor.execute("SHOW TABLES")
print(cursor.fetchall())

cursor.close()
conn.close()

print("\nDONE 🚀")

Files Found: ['store_sales_1.csv', 'store_sales_2.csv', 'store_sales_3.csv']
store_sales_1 loaded ✔
store_sales_2 loaded ✔
store_sales_3 loaded ✔
Connected ✔

Processing store_sales_1 ...
Table store_sales_1_processed created ✔
Data inserted ✔

Processing store_sales_2 ...
Table store_sales_2_processed created ✔
Data inserted ✔

Processing store_sales_3 ...
Table store_sales_3_processed created ✔
Data inserted ✔
[('store_sales_1_processed',), ('store_sales_2_processed',), ('store_sales_3_processed',)]

DONE 🚀
